# panoptic-prelabel: GPU benchmarks (Kaggle)

Runs the three benchmarks of `benchmarks/` on a GPU and packs the results in one zip to download.
Read `benchmarks/README.md` first (how to upload the repository and set up the notebook).

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**. Total run time: roughly 1 to 1.5 hours,
most of it in TensorRT engine builds and the COCO download.

In [ ]:
# 1. Copy the repository (uploaded as a Kaggle dataset) to a writable folder
import glob
import os
import shutil

src = sorted(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True))
assert src, "Add the dataset holding the repository zip: right panel > Add Input"
repo = "/kaggle/working/panoptic-prelabel"
shutil.copytree(os.path.dirname(src[0]), repo, dirs_exist_ok=True)
os.chdir(repo)
# Kaggle unpacks the nested examples/*/cvat_corrected.zip into folders: zip them back
for d in glob.glob("examples/*/cvat_corrected") + glob.glob("examples/*/cvat_corrected.zip"):
    if os.path.isdir(d):
        target = os.path.join(os.path.dirname(d), "cvat_corrected")
        tmp = shutil.make_archive(target + "_rezip", "zip", d)  # keeps annotations/instances_default.json
        shutil.rmtree(d)
        os.replace(tmp, target + ".zip")
zips = glob.glob("examples/*/cvat_corrected.zip")
assert len(zips) == 4 and all(os.path.isfile(z) for z in zips), zips
print(os.getcwd(), os.listdir("."))

In [ ]:
# 2. The GPU, and the libraries matching the preinstalled PyTorch (onnxruntime-gpu, TensorRT, ...)
!nvidia-smi
# --no-deps: keep Kaggle's own numpy / OpenCV / SciPy / Pillow builds
!pip install -q --no-deps -e .
!python -c "import panoptic_prelabel, cv2, numpy, scipy, yaml; print('package ok', numpy.__version__, cv2.__version__)"
!python benchmarks/setup_gpu_env.py

In [ ]:
# 3. Model weights (sha256-checked) and the test suite, including the parity test with panopticapi
!panoptic-prelabel download-weights
!python -m pytest -q

## Benchmark 1: latency, CPU vs GPU

Mask R-CNN through onnxruntime (CPU, CUDA, TensorRT FP32 / FP16), MobileSAM automatic masks (CPU, CUDA),
and which parts of the ONNX graph TensorRT supports. Each GPU result is checked against the CPU one.

In [ ]:
!python benchmarks/latency.py --name kaggle

## Benchmark 2: Mask2Former vs this pipeline, on the 4 example photos

In [ ]:
!python benchmarks/mask2former_baseline.py --name kaggle

## Benchmark 3: Mask2Former on 100 COCO val2017 images, scored by panopticapi and by `compare`

Downloads the COCO panoptic annotations (~820 MB) and 100 images into /tmp (not kept as notebook output).

In [ ]:
!python benchmarks/coco_val.py --name kaggle --images 100 --cache /tmp/coco_cache

In [ ]:
# 4. Pack the results: download /kaggle/working/results_kaggle.zip from the Output panel
import os
import shutil

os.chdir("/kaggle/working/panoptic-prelabel")  # also after a kernel restart
shutil.make_archive("/kaggle/working/results_kaggle", "zip", "benchmarks/results")
print(
    open("benchmarks/results/kaggle/latency.md").read()[:3000]
    if os.path.exists("benchmarks/results/kaggle/latency.md")
    else "no latency results"
)